In [1]:
# Installs / M
%pip install -q easyocr pyyaml natsort lmdb python-bidi pandas nltk matplotlib opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# GPU Check / M2
import torch
print(torch.__version__, torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - fix PyTorch install first")
!nvidia-smi

2.11.0+cu128 True
NVIDIA GeForce RTX 4050 Laptop GPU
Sat Nov  7 18:48:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 621.14                 KMD Version: 621.14        CUDA UMD Version: 13.5     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   43C    P8              2W /   50W |     228MiB /   6141MiB |     18%      Default |
|                                         |                        |                  N

In [4]:
# # paths, clone, patches, dataset copy/ M3
import os, sys, subprocess, shutil
from pathlib import Path

WORK     = Path(r"D:\Projects\MediLens")
DATA_SRC = WORK / "easyocr_data"
EASYOCR  = WORK / "EasyOCR"
TRAINER  = EASYOCR / "trainer"
config_path = TRAINER / "config_files" / "medical_prescription.yaml"
RESULTS = WORK / "results"; RESULTS.mkdir(exist_ok=True)
MODELS  = WORK / "models";  MODELS.mkdir(exist_ok=True)

# clone EasyOCR once
if not EASYOCR.exists():
    subprocess.run(["git", "clone", "https://github.com/JaidedAI/EasyOCR.git", str(EASYOCR)], check=True)

# patch dataset.py (newer torch compatibility)
f = TRAINER / "dataset.py"
t = f.read_text(encoding="utf-8")
t = t.replace("from torch._utils import _accumulate", "from itertools import accumulate as _accumulate")
t = t.replace("data_loader_iter.next()", "next(data_loader_iter)")
t = t.replace("self.dataloader_iter_list[i].next()", "next(self.dataloader_iter_list[i])")
f.write_text(t, encoding="utf-8")

# patch test.py (all tensors on one device for CTC loss)
f = TRAINER / "test.py"
t = f.read_text(encoding="utf-8")
old = "cost = criterion(preds.log_softmax(2).permute(1, 0, 2), text_for_loss, preds_size, length_for_loss)"
new = ("cost = criterion(preds.log_softmax(2).permute(1, 0, 2), "
       "text_for_loss.to(device), preds_size.to(device), length_for_loss.to(device))")
if old in t:
    f.write_text(t.replace(old, new), encoding="utf-8"); print("Patched test.py")
elif new in t:
    print("test.py already patched")
else:
    raise RuntimeError("Expected criterion line not found in test.py")

# report any remaining old-style .next() calls
for p in TRAINER.glob("*.py"):
    for i, line in enumerate(p.read_text(encoding="utf-8").splitlines(), 1):
        if ".next()" in line:
            print("Still present:", p.name, i, line.strip())

# copy the dataset ONCE. Never overwrite: all_data also holds the augmented and held-out folders
dst = TRAINER / "all_data"
if dst.exists():
    print("all_data already exists -> NOT copying again")
else:
    shutil.copytree(DATA_SRC, dst)
for d in sorted(dst.iterdir()):
    print(d.name, len(list(d.glob("*.png"))), "images")

test.py already patched
all_data already exists -> NOT copying again
holdout_train 14000 images
holdout_val 700 images
kaggle_f0_test 18 images
kaggle_f0_train 51 images
kaggle_f0_train_aug 816 images
kaggle_f0_val 18 images
kaggle_f1_test 18 images
kaggle_f1_train 52 images
kaggle_f1_train_aug 832 images
kaggle_f1_val 17 images
kaggle_f2_test 17 images
kaggle_f2_train 53 images
kaggle_f2_train_aug 848 images
kaggle_f2_val 17 images
kaggle_f3_test 17 images
kaggle_f3_train 53 images
kaggle_f3_train_aug 848 images
kaggle_f3_val 17 images
kaggle_f4_test 17 images
kaggle_f4_train 52 images
kaggle_f4_train_aug 832 images
kaggle_f4_val 18 images
medical_test 780 images
medical_train 3120 images
medical_train_aug 15600 images
medical_val 780 images


In [8]:
#import-fix / M4
import sys, os
from pathlib import Path

WORK    = Path(r"D:\Projects\MediLens")
TRAINER = WORK / "EasyOCR" / "trainer"
config_path = TRAINER / "config_files" / "medical_prescription.yaml"
RESULTS = WORK / "results"; RESULTS.mkdir(exist_ok=True)
MODELS  = WORK / "models";  MODELS.mkdir(exist_ok=True)

# drop any wrongly-resolved modules, then put the trainer first on the path
for k in ("utils", "model", "dataset", "test", "train"):
    sys.modules.pop(k, None)
sys.path.insert(0, str(TRAINER))
os.chdir(TRAINER)

import utils
print(utils.__file__)                          # must end with ...\trainer\utils.py
from utils import AttrDict, CTCLabelConverter

D:\Projects\MediLens\EasyOCR\trainer\utils.py


In [5]:
# Config / Cell 8
import easyocr, yaml, torch

easyocr.Reader(['en'], gpu=True)   # downloads english_g2.pth on first run
pre_path = Path.home() / ".EasyOCR" / "model" / "english_g2.pth"
assert pre_path.exists(), pre_path

number    = '0123456789'
symbol    = "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ €"
lang_char = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
n_classes = len(number + symbol + lang_char) + 1

sd = torch.load(pre_path, map_location="cpu")
head_key = next(k for k in sd if k.endswith("Prediction.weight"))
pre_classes = sd[head_key].shape[0]
heads_match = (pre_classes == n_classes)
print("pretrained classes:", pre_classes, "| yours:", n_classes, "| match:", heads_match)

cfg = dict(
    number=number, symbol=symbol, lang_char=lang_char,
    experiment_name='medical_prescription',
    train_data='all_data', valid_data='all_data/medical_val',
    manualSeed=1111, workers=1, batch_size=32, num_iter=10000, valInterval=500,
    saved_model=str(pre_path), FT=True,
    optim=False, lr=1.0, beta1=0.9, rho=0.95, eps=1e-8, grad_clip=5,
    select_data='medical_train', batch_ratio='1',
    total_data_usage_ratio=1.0, batch_max_length=64,
    imgH=64, imgW=600, rgb=False, contrast_adjust=False,
    sensitive=True, PAD=True, data_filtering_off=False,
    Transformation='None', FeatureExtraction='VGG',
    SequenceModeling='BiLSTM', Prediction='CTC',
    num_fiducial=20, input_channel=1, output_channel=256, hidden_size=256,
    decode='greedy', new_prediction=(not heads_match),
    freeze_FeatureFxtraction=False, freeze_SequenceModeling=False,
)

config_path = TRAINER / "config_files" / "medical_prescription.yaml"
config_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
print("Wrote", config_path)

pretrained classes: 97 | yours: 97 | match: True
Wrote D:\Projects\MediLens\EasyOCR\trainer\config_files\medical_prescription.yaml


In [6]:
# Data Sanity / M5
import pandas as pd
base = TRAINER / "all_data"

for split in ["medical_train", "medical_val", "medical_test"]:
    d = base / split
    chk = pd.read_csv(d / "labels.csv")
    assert list(chk.columns) == ["filename", "words"]
    assert len(chk) == len(list(d.glob("*.png")))
    assert all((d / n).exists() for n in chk["filename"])
    print(split, "OK:", len(chk), "samples")

tr = pd.read_csv(base/"medical_train"/"labels.csv")["words"]
va = pd.read_csv(base/"medical_val"/"labels.csv")["words"]
te = pd.read_csv(base/"medical_test"/"labels.csv")["words"]
print("unique names  train/val/test:", tr.nunique(), va.nunique(), te.nunique())
print("val rows with name seen in train: %.1f%%" % (va.isin(set(tr)).mean()*100))
print("test rows with name seen in train: %.1f%%" % (te.isin(set(tr)).mean()*100))

medical_train OK: 3120 samples
medical_val OK: 780 samples
medical_test OK: 780 samples
unique names  train/val/test: 78 78 78
val rows with name seen in train: 100.0%
test rows with name seen in train: 100.0%


In [ ]:
# Train Run 1 / M6
import os, yaml
import torch.backends.cudnn as cudnn
from train import train
from utils import AttrDict

with open(config_path, "r", encoding="utf-8") as fh:
    opt = AttrDict(yaml.safe_load(fh))
opt.character = opt.number + opt.symbol + opt.lang_char
os.makedirs(f"./saved_models/{opt.experiment_name}", exist_ok=True)

cudnn.benchmark = True
cudnn.deterministic = False

try:
    train(opt, amp=True)
except SystemExit:          # the trainer calls sys.exit() when it finishes
    pass

In [2]:
# import cv2, random, shutil, augmentation / Cell 13a
import cv2, random, shutil
import numpy as np, pandas as pd
from pathlib import Path

def augment(img):
    h, w = img.shape[:2]
    ang, shear, scale = random.uniform(-3, 3), random.uniform(-0.12, 0.12), random.uniform(0.92, 1.05)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, scale)
    M[0, 1] += shear
    M[0, 2] -= shear * h / 2
    out = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    if random.random() < 0.5:
        k = random.choice([3, 5])
        out = cv2.GaussianBlur(out, (k, k), 0)
    if random.random() < 0.4:
        op = cv2.erode if random.random() < 0.5 else cv2.dilate
        out = op(out, np.ones((2, 2), np.uint8))
    out = cv2.convertScaleAbs(out, alpha=random.uniform(0.75, 1.25), beta=random.uniform(-25, 25))
    noise = np.random.normal(0, random.uniform(0, 10), out.shape)
    out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if random.random() < 0.3:
        _, buf = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, random.randint(35, 75)])
        out = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return out

In [ ]:
# (do no run unless regenrating the old aug is what i want)import cv2, random, shutil, augmentation / Cell 13b

random.seed(0); np.random.seed(0)
SRC = TRAINER / "all_data" / "medical_train"
DST = TRAINER / "all_data" / "medical_train_aug"
N_AUG = 4

if DST.exists():
    shutil.rmtree(DST)
DST.mkdir()



df_tr = pd.read_csv(SRC / "labels.csv")
rows = []
for fn, word in zip(df_tr["filename"], df_tr["words"]):
    shutil.copy2(SRC / fn, DST / fn)
    rows.append((fn, word))
    img = cv2.imread(str(SRC / fn), cv2.IMREAD_COLOR)
    assert img is not None, f"could not read {fn}"
    for i in range(N_AUG):
        new = f"{Path(fn).stem}_aug{i}.png"
        cv2.imwrite(str(DST / new), augment(img))
        rows.append((new, word))

pd.DataFrame(rows, columns=["filename", "words"]).to_csv(DST / "labels.csv", index=False)
print(len(rows), "training images written to", DST)

In [ ]:
# Retrain Augmented + Backup / M7
import gc, shutil, yaml, torch, os
import pandas as pd
from train import train
from utils import AttrDict

# free any eval model and back up run 1
globals().pop("model", None)
gc.collect(); torch.cuda.empty_cache()
shutil.copy2(TRAINER/"saved_models"/"medical_prescription"/"best_accuracy.pth", MODELS/"run1_best_accuracy.pth")

# fresh opt from the config file
with open(config_path, encoding="utf-8") as fh:
    opt = AttrDict(yaml.safe_load(fh))
opt.character    = opt.number + opt.symbol + opt.lang_char
opt.workers      = 1
opt.select_data  = 'medical_train_aug'
opt.experiment_name = 'medical_prescription_aug'
opt.lr           = 0.3
opt.num_iter     = 5000
opt.valInterval  = 250

aug = TRAINER/"all_data"/"medical_train_aug"
print("aug images:", len(list(aug.glob("*.png"))), "| rows:", len(pd.read_csv(aug/"labels.csv")))
print("start from:", opt.saved_model, "| FT:", opt.FT, "| new_prediction:", opt.new_prediction)

os.chdir(TRAINER)
os.makedirs(f"./saved_models/{opt.experiment_name}", exist_ok=True)
try:
    train(opt, amp=True)
except SystemExit:
    pass

# back up the final model
shutil.copy2(TRAINER/"saved_models"/"medical_prescription_aug"/"best_accuracy.pth", MODELS/"final_best_accuracy.pth")

In [7]:
# Held-out split / Cell 20
import random, shutil
import pandas as pd
from nltk.metrics.distance import edit_distance

base = TRAINER / "all_data"
names = sorted(set(pd.read_csv(base/"medical_train_aug"/"labels.csv")["words"]))

NOISY = {"Fenadin","Fexo","Fixal","Dinafex","Nexcap","Opton","Nexum","Sergel",
         "Maxpro","Exium","Montair","Montene","Bacaid"}

def ok(n):
    nl = n.lower()
    others = [m.lower() for m in names if m != n]
    if any(nl in m or m in nl for m in others):
        return False
    return min(edit_distance(nl, m) for m in others) >= 3

cands = [n for n in names if n not in NOISY and ok(n)]
print(len(cands), "eligible names")
assert len(cands) >= 8, "too few eligible names; relax the filter"

HOLDOUT = sorted(random.Random(0).sample(cands, 8))   # change the seed to repeat with other names
hold = set(HOLDOUT)
print("held-out names:", HOLDOUT)

def make_split(src_dir, dst_dir, keep_fn):
    df = pd.read_csv(src_dir/"labels.csv")
    df = df[df["words"].map(keep_fn)]
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True)
    for fn in df["filename"]:
        shutil.copy2(src_dir/fn, dst_dir/fn)
    df.to_csv(dst_dir/"labels.csv", index=False)
    return len(df)

n_tr = make_split(base/"medical_train_aug", base/"holdout_train", lambda w: w not in hold)
n_va = make_split(base/"medical_val",       base/"holdout_val",   lambda w: w not in hold)
print("holdout_train:", n_tr, "| holdout_val:", n_va)

for d in ("holdout_train", "holdout_val"):
    assert not (set(pd.read_csv(base/d/"labels.csv")["words"]) & hold), d
print("no held-out name leaked into training or validation")

38 eligible names
held-out names: ['Bacmax', 'Lucan-R', 'Progut', 'Provair', 'Renova', 'Rivotril', 'Rozith', 'Tamen']
holdout_train: 14000 | holdout_val: 700
no held-out name leaked into training or validation


In [ ]:
# Held-out training / Cell 21
from train import train
import gc, os, yaml, torch
from utils import AttrDict

globals().pop("model", None); gc.collect(); torch.cuda.empty_cache()

with open(config_path, encoding="utf-8") as fh:
    opt = AttrDict(yaml.safe_load(fh))
opt.character       = opt.number + opt.symbol + opt.lang_char
opt.workers         = 1
opt.select_data     = "holdout_train"
opt.valid_data      = "all_data/holdout_val"
opt.experiment_name = "holdout"
opt.lr              = 0.3
opt.num_iter        = 5000
opt.valInterval     = 250

print("starts from:", opt.saved_model, "| FT:", opt.FT, "| new_prediction:", opt.new_prediction)
assert "english_g2" in str(opt.saved_model) and opt.FT, "config was overwritten; do not train"

os.chdir(TRAINER)
os.makedirs(f"./saved_models/{opt.experiment_name}", exist_ok=True)
try:
    train(opt, amp=True)
except SystemExit:
    pass

In [8]:
# Write Vocab / Cell 23
import pandas as pd
names = sorted(set(pd.read_csv(TRAINER/"all_data"/"medical_train"/"labels.csv")["words"]))
Path(r"D:\Projects\MediLens\app\vocab.txt").write_text("\n".join(names), encoding="utf-8")

571

In [11]:
# Augment Kaggle fold training sets (uses augment from Cell 13a) / Cell 24
random.seed(1); np.random.seed(1)
K, N_AUG_KAGGLE = 5, 15
base = TRAINER / "all_data"

# data for the final model: all 87 Kaggle images = the five fold test splits (each image is in exactly one)
all_tr = base / "kaggle_fall_train"
if all_tr.exists():
    shutil.rmtree(all_tr)
all_tr.mkdir()
rows_all = []
for j in range(K):
    d = pd.read_csv(base / f"kaggle_f{j}_test" / "labels.csv")
    for fn, word in zip(d["filename"], d["words"]):
        shutil.copy2(base / f"kaggle_f{j}_test" / fn, all_tr / fn)
        rows_all.append((fn, word))
pd.DataFrame(rows_all, columns=["filename", "words"]).to_csv(all_tr / "labels.csv", index=False)
print("final-model images:", len(rows_all))      # expect 87

# the trainer needs a validation folder; this one is a formality (the model trains on these images)
all_val = base / "kaggle_fall_val"
if all_val.exists():
    shutil.rmtree(all_val)
shutil.copytree(base / "kaggle_f0_val", all_val)

for k in ["all"]:
    src, dst = base / f"kaggle_f{k}_train", base / f"kaggle_f{k}_train_aug"
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir()
    d = pd.read_csv(src / "labels.csv")
    rows = []
    for fn, word in zip(d["filename"], d["words"]):
        shutil.copy2(src / fn, dst / fn); rows.append((fn, word))
        img = cv2.imread(str(src / fn), cv2.IMREAD_COLOR)
        assert img is not None, fn
        for i in range(N_AUG_KAGGLE):
            new = f"{Path(fn).stem}_aug{i}.png"
            cv2.imwrite(str(dst / new), augment(img)); rows.append((new, word))
    pd.DataFrame(rows, columns=["filename", "words"]).to_csv(dst / "labels.csv", index=False)
    print(f"fold {k}: {len(rows)} training images")

final-model images: 87
fold all: 1392 training images


In [12]:
# Fine-tune one model per fold / Cell 25
import gc, os, shutil, yaml, torch
from train import train
from utils import AttrDict

START = MODELS / "final_best_accuracy.pth"
assert START.exists(), START
os.chdir(TRAINER)

# Change the line back to range(5) if i ever rerun the folds.
for k in ["all"]:
    globals().pop("model", None); gc.collect(); torch.cuda.empty_cache()
    with open(config_path, encoding="utf-8") as fh:
        opt = AttrDict(yaml.safe_load(fh))
    opt.character       = opt.number + opt.symbol + opt.lang_char
    opt.workers         = 1
    opt.select_data     = f"medical_train_aug-kaggle_f{k}_train_aug"
    opt.batch_ratio     = "0.5-0.5"
    opt.valid_data      = f"all_data/kaggle_f{k}_val"
    opt.experiment_name = f"kaggle_e400_f{k}"
    opt.saved_model     = str(START)
    opt.FT              = True
    opt.new_prediction  = False
    opt.lr              = 0.1
    opt.num_iter        = 400
    opt.valInterval     = 400
    print(f"\n===== fold {k} =====", opt.select_data, "| start:", START.name)
    os.makedirs(f"./saved_models/{opt.experiment_name}", exist_ok=True)
    try:
        train(opt, amp=True)
    except SystemExit:
        pass
    shutil.copy2(TRAINER / "saved_models" / opt.experiment_name / "best_accuracy.pth",
                 MODELS / f"kaggle_e400_f{k}_best.pth")


===== fold all ===== medical_train_aug-kaggle_fall_train_aug | start: final_best_accuracy.pth
Filtering the images containing characters which are not in opt.character
Filtering the images whose label is longer than opt.batch_max_length
--------------------------------------------------------------------------------
dataset_root: all_data
opt.select_data: ['medical_train_aug', 'kaggle_fall_train_aug']
opt.batch_ratio: ['0.5', '0.5']
--------------------------------------------------------------------------------
dataset_root:    all_data	 dataset: medical_train_aug
all_data/medical_train_aug
sub-directory:	/medical_train_aug	 num samples: 15600
num total samples of medical_train_aug: 15600 x 1.0 (total_data_usage_ratio) = 15600
num samples of medical_train_aug per batch: 32 x 0.5 (batch_ratio) = 16
--------------------------------------------------------------------------------
dataset_root:    all_data	 dataset: kaggle_fall_train_aug
all_data/kaggle_fall_train_aug
sub-directory:	/kag

D:\Projects\MediLens\EasyOCR\trainer\train.py:172: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
D:\Projects\MediLens\EasyOCR\trainer\train.py:180: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


training time:  109.79948949813843
[400/400] Train loss: 0.63035, Valid loss: 0.26982, Elapsed_time: 109.79949
Current_accuracy : 55.556, Current_norm_ED  : 0.9477
Best_accuracy    : 55.556, Best_norm_ED     : 0.9477
--------------------------------------------------------------------------------
Ground Truth              | Prediction                | Confidence Score & T/F
--------------------------------------------------------------------------------
Tab Atcomid               | ab Atcomid                | 0.1270	False
Tab Montiget 10mg         | Tab Montget10m            | 0.0053	False
--------------------------------------------------------------------------------
validation time:  5.934916973114014
end the training
